<a href="https://colab.research.google.com/github/Bovi-analytics/Ansci4040FinalClassProject/blob/main/NamingConvention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from google.colab import drive
from pathlib import Path
import re
import unicodedata
import uuid

In [ ]:

# ==========================================
# 1. CONNECT GOOGLE DRIVE
# ==========================================

drive.mount('/content/drive')

# CHANGE THIS to your project folder
PROJECT_DIR = Path(
    "/content/drive/MyDrive/ANSC4040GroupProject"
)


In [ ]:


# ==========================================
# 2. NAMING CONVENTION
# ==========================================

def standardize_name(filename):


    path = Path(filename)

    # Preserve all extensions
    extensions = "".join(path.suffixes)
    stem = filename[:-len(extensions)] if extensions else filename

    # Normalize text
    stem = unicodedata.normalize("NFKD", stem)
    stem = stem.encode("ascii", "ignore").decode("ascii")

    # Split words based on spaces, underscores,
    # hyphens, and other punctuation
    words = re.findall(r"[A-Za-z0-9]+", stem)

    # Convert to PascalCase
    new_stem = "".join(
        word[0].upper() + word[1:].lower()
        for word in words
        if word
    )

    return new_stem + extensions.lower()


In [ ]:
# ==========================================
# 3. FIND ALL FILES
# ==========================================

files = [
    p for p in PROJECT_DIR.rglob("*")
    if p.is_file() and not p.is_symlink()
]

rename_plan = []

for file in files:
    new_name = standardize_name(file.name)
    new_path = file.parent / new_name

    if file != new_path:
        rename_plan.append((file, new_path))


In [ ]:

# ==========================================
# 4. PREVIEW CHANGES
# ==========================================

print(f"Total files found: {len(files)}")
print(f"Files requiring renaming: {len(rename_plan)}")

print("\nPROPOSED RENAMES:\n")

for old, new in rename_plan:
    print(f"{old.relative_to(PROJECT_DIR)}")
    print(f"    --> {new.name}\n")


In [ ]:
# ==========================================
# 5. CHECK FOR NAME CONFLICTS
# ==========================================

sources = {old for old, new in rename_plan}
targets = [new for old, new in rename_plan]

if len(targets) != len(set(targets)):
    raise ValueError("Duplicate destination filenames detected.")

for target in targets:
    if target.exists() and target not in sources:
        raise FileExistsError(
            f"Destination already exists: {target}"
        )

print("No naming conflicts detected.")


In [ ]:
# ==========================================
# 6. APPLY RENAMING
# ==========================================

# IMPORTANT:
# Leave False until you have reviewed the preview.
APPLY_CHANGES = False

if APPLY_CHANGES:

    # First move files to temporary names to avoid
    # conflicts when two files exchange names.
    temporary_moves = []

    for old, new in rename_plan:
        temp = old.with_name(
            f".rename_temp_{uuid.uuid4().hex}"
        )
        old.rename(temp)
        temporary_moves.append((temp, new))

    # Move files to their final destinations
    for temp, new in temporary_moves:
        temp.rename(new)

    print(f"\nSuccessfully renamed {len(rename_plan)} files!")

else:
    print("\nPREVIEW ONLY. No files have been renamed.")
    print("Change APPLY_CHANGES to True to execute.")
